<a href="https://colab.research.google.com/github/cdascientist/Vis/blob/master/vis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [93]:
"""
time_factory_pipeline.py
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
TANDEM FACTORY PIPELINE  ·  JIT Object Instantiation Template
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

Both objects carry a nested hierarchy of sections that grow as they move
through the pipeline.  Each phase appends its own section onto both objects.

Object hierarchy shape:

  InboundRequestObject
  └── .caller          ← populated by caller before factory call
  └── .phase1          ← appended by Phase 1
  └── .phase2          ← appended by Phase 2
  └── .phase3          ← appended by Phase 3
  └── .phase4          ← appended by Phase 4
  └── .telemetry       ← attached at factory entry

  JITWorkObject
  └── .phase1          ← appended by Phase 1
  └── .phase2          ← appended by Phase 2
  └── .phase3          ← appended by Phase 3
  └── .phase4          ← appended by Phase 4
"""

import datetime
import pytz
import os
import logging
import ctypes
import gc
import time
from dataclasses import dataclass, field
from typing import Optional, Any


# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
#  EXECUTION MARKER HELPERS
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

def ms_now() -> str:
    """Wall-clock time precise to the millisecond.  YYYY-MM-DD  HH:MM:SS.mmm"""
    now = datetime.datetime.now()
    return now.strftime("%Y-%m-%d  %H:%M:%S.") + f"{now.microsecond // 1000:03d}"

def obj_uid(obj: object) -> str:
    """Unique in-memory execution ID.  OBJ#<hex_address>@<ref_count>"""
    addr = hex(ctypes.addressof(ctypes.c_int.from_address(id(obj))))
    refs = ctypes.c_long.from_address(id(obj)).value
    return f"OBJ#{addr}@{refs}"

def stamp(obj: object, event: str = "") -> str:
    """Single-line execution marker:  [ ms ]  OBJ#addr@refs  event"""
    return f"[ {ms_now()} ]  {obj_uid(obj)}  {event}".rstrip()


# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
#  HIERARCHICAL SECTION DATACLASSES
#  Each section is a plain dataclass.  Instances are appended onto both
#  objects by the phase that owns them.  The log renderer walks the sections
#  in order to produce the hierarchical output.
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

@dataclass
class CallerSection:
    """Populated by the caller before the factory is invoked."""
    request_label   : Optional[str] = None
    # ↑  TEMPLATE : add your own domain fields here
    entry_marker    : Optional[str] = None    # stamped at invocation

@dataclass
class Phase1Section:
    """Appended to both objects by Phase 1 — acquisition."""
    entry_marker        : Optional[str]               = None
    # JIT-side data (raw acquired objects)
    timezone_object     : Optional[object]            = None
    raw_utc_datetime    : Optional[datetime.datetime] = None
    utc_tzinfo          : Optional[str]               = None
    raw_local_datetime  : Optional[datetime.datetime] = None
    local_tzinfo        : Optional[str]               = None
    utc_offset          : Optional[object]            = None
    calendar_year       : Optional[int]               = None
    calendar_month      : Optional[int]               = None
    calendar_day        : Optional[int]               = None
    clock_hour          : Optional[int]               = None
    clock_minute        : Optional[int]               = None
    clock_second        : Optional[int]               = None
    clock_microsecond   : Optional[int]               = None
    # Inbound-side data (receipt confirmation)
    pipeline_received   : Optional[bool]              = None
    exit_marker         : Optional[str]               = None

@dataclass
class Phase2Section:
    """Appended to both objects by Phase 2 — transformation + transfer."""
    entry_marker          : Optional[str]  = None
    formatted_gmt_stamp   : Optional[str]  = None
    formatted_local_stamp : Optional[str]  = None
    # Transfer moment — same marker on both objects
    transfer_marker       : Optional[str]  = None
    # Inbound-side: human-readable copies transferred from JIT
    acquired_utc_str      : Optional[str]  = None
    acquired_local_str    : Optional[str]  = None
    exit_marker           : Optional[str]  = None

@dataclass
class Phase3Section:
    """Appended to both objects by Phase 3 — output confirmation."""
    entry_marker      : Optional[str]  = None
    display_written   : Optional[bool] = None   # JIT side
    display_confirmed : Optional[bool] = None   # Inbound side
    display_marker    : Optional[str]  = None   # same ms on both
    exit_marker       : Optional[str]  = None

@dataclass
class Phase4Section:
    """Appended to both objects by Phase 4 — log + release."""
    entry_marker      : Optional[str]          = None
    log_file_path     : Optional[str]          = None
    log_write_marker  : Optional[str]          = None   # same ms on both
    log_confirmed     : Optional[bool]         = None   # Inbound side
    release_marker    : Optional[str]          = None   # JIT side — final marker
    exit_marker       : Optional[str]          = None   # Inbound side only


# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
#  TELEMETRY
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

@dataclass
class PipelineTelemetry:
    pipeline_start_ns : Optional[int] = None
    phase_entry_ns    : dict = field(default_factory=dict)
    phase_exit_ns     : dict = field(default_factory=dict)
    phase_duration_ns : dict = field(default_factory=dict)
    feature_log       : list = field(default_factory=list)  # ordered list of (ms, step, feature, target, value)

    def phase_enter(self, name: str) -> None:
        self.phase_entry_ns[name] = time.perf_counter_ns()

    def phase_exit(self, name: str) -> None:
        self.phase_exit_ns[name]    = time.perf_counter_ns()
        self.phase_duration_ns[name] = self.phase_exit_ns[name] - self.phase_entry_ns.get(name, 0)

    def record(self, step: str, feature: str, target: str, value: Any) -> None:
        self.feature_log.append((ms_now(), step, feature, target, str(value)[:100]))

    def ms_duration(self, name: str) -> str:
        ns = self.phase_duration_ns.get(name, 0)
        return f"{ns / 1_000_000:.3f} ms"


# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
#  INBOUND REQUEST OBJECT
#  Carries domain context into the pipeline.
#  Each phase appends its own section.  Returned to caller fully populated.
#  TEMPLATE : rename to your domain (OrderRequest, SensorPayload, etc.)
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

@dataclass
class InboundRequestObject:
    caller    : CallerSection              = field(default_factory=CallerSection)
    phase1    : Optional[Phase1Section]    = None
    phase2    : Optional[Phase2Section]    = None
    phase3    : Optional[Phase3Section]    = None
    phase4    : Optional[Phase4Section]    = None
    telemetry : Optional[PipelineTelemetry]= None


# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
#  JIT WORK OBJECT
#  Created bare inside the factory.  Each phase appends its own section.
#  Destroyed manually at end of Phase 4.
#  TEMPLATE : rename to your domain (JITPriceCalc, JITSensorReader, etc.)
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

@dataclass
class JITWorkObject:
    phase1 : Optional[Phase1Section] = None
    phase2 : Optional[Phase2Section] = None
    phase3 : Optional[Phase3Section] = None
    phase4 : Optional[Phase4Section] = None


# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
#  LOG RENDERER
#  _L()                  — write one line to both console and log file
#  _section_fields()     — collect populated fields from a section dataclass
#  _render_section()     — draw one named section with ├─ / └─ connectors
#  render_log_header()   — clean sequential header written once at pipeline start
#  render_object_hierarchy() — full hierarchical tree of both objects
#  render_telemetry_summary()— numbered sequential telemetry report
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

def _L(logger: logging.Logger, line: str = "") -> None:
    """Write one line to both console and log file."""
    print(line)
    logger.info(line)

def _section_fields(section) -> list[tuple[str, str]]:
    """Return populated (name, str-value) pairs from any section dataclass."""
    return [(k, str(v)) for k, v in section.__dict__.items() if v is not None]

def _render_section(logger: logging.Logger, section, section_name: str,
                    is_last_section: bool, indent: str = "  │  ") -> None:
    """
    Render one named section with correct ├─ / └─ tree connectors.

      indent ├─ section_name
      indent │  ├─ field  :  value
      indent │  └─ field  :  value   (last field gets └─)
      indent │                        (spacer bar after non-last sections)
    """
    section_conn = "└─" if is_last_section else "├─"
    child_bar    = "   " if is_last_section else "│  "

    _L(logger, f"{indent}{section_conn} {section_name}")

    fields = _section_fields(section)
    for i, (k, v) in enumerate(fields):
        field_conn = "└─" if i == len(fields) - 1 else "├─"
        _L(logger, f"{indent}{child_bar}  {field_conn} {k:<26}  {v}")

    if not is_last_section:
        _L(logger, f"{indent}{child_bar}")


def render_log_header(logger: logging.Logger,
                      jit: JITWorkObject,
                      inbound: InboundRequestObject) -> None:
    """
    Clean sequential header written once — the very first thing in the log file.
    Shows exactly what entered the pipeline and what phases will run.
    No noise.  No repeated decorators.  Easy to read from top to bottom.
    """
    _L(logger)
    _L(logger, "  ╔══════════════════════════════════════════════════════════════")
    _L(logger, "  ║  PIPELINE EXECUTION LOG")
    _L(logger, f"  ║  {ms_now()}")
    _L(logger, "  ╠══════════════════════════════════════════════════════════════")
    _L(logger)
    _L(logger, "  ║  1.  What was passed in")
    _L(logger, f"  ║        InboundRequestObject   {obj_uid(inbound)}")
    _L(logger, f"  ║          caller.request_label  :  {inbound.caller.request_label}")
    _L(logger, f"  ║          caller.entry_marker   :  {inbound.caller.entry_marker}")
    _L(logger)
    _L(logger, "  ║  2.  What was created inside the factory")
    _L(logger, f"  ║        JITWorkObject (bare)   {obj_uid(jit)}")
    _L(logger,  "  ║          no sections yet — each phase will append one")
    _L(logger)
    _L(logger, "  ║  3.  Phases that will run in order")
    _L(logger, "  ║        Phase 1  ·  Acquisition")
    _L(logger, "  ║        Phase 2  ·  Transformation + Transfer")
    _L(logger, "  ║        Phase 3  ·  Output Confirmation")
    _L(logger, "  ║        Phase 4  ·  Log + Release JIT")
    _L(logger)
    _L(logger, "  ╚══════════════════════════════════════════════════════════════")
    _L(logger)


def render_object_hierarchy(logger: logging.Logger,
                             jit: JITWorkObject,
                             inbound: InboundRequestObject,
                             title: str = "Object State") -> None:
    """
    Render the full hierarchical state of both objects.

    Layout (one clean tree per object, separated by a ruled divider):

      ┌─  <title>  ·  <timestamp>
      │
      │  JITWorkObject   OBJ#addr@refs
      │  ├─ phase1
      │  │  ├─ field_name              value
      │  │  └─ field_name              value
      │  │
      │  └─ phase4
      │     └─ field_name              value
      │
      │  ──────────────────────────────────────
      │
      │  InboundRequestObject   OBJ#addr@refs
      │  ├─ caller
      │  │  └─ field_name              value
      │  │
      │  └─ phase4
      │     └─ field_name              value
      │
      └──────────────────────────────────────
    """
    _L(logger)
    _L(logger, f"  ┌─────────────────────────────────────────────────────────────")
    _L(logger, f"  │  {title}")
    _L(logger, f"  │  {ms_now()}")
    _L(logger)

    # ── JIT Work Object ───────────────────────────────────────────────────────
    _L(logger, f"  │  JITWorkObject   {obj_uid(jit)}")
    _L(logger,  "  │  │")

    jit_sections = [n for n in ("phase1","phase2","phase3","phase4")
                    if getattr(jit, n) is not None]
    for i, name in enumerate(jit_sections):
        _render_section(logger, getattr(jit, name), name,
                        is_last_section=(i == len(jit_sections) - 1),
                        indent="  │  ")

    _L(logger)
    _L(logger,  "  │  ─────────────────────────────────────────────────────────")
    _L(logger)

    # ── Inbound Request Object ────────────────────────────────────────────────
    _L(logger, f"  │  InboundRequestObject   {obj_uid(inbound)}")
    _L(logger,  "  │  │")

    inbound_sections = ["caller"] + [
        n for n in ("phase1","phase2","phase3","phase4")
        if getattr(inbound, n) is not None
    ]
    for i, name in enumerate(inbound_sections):
        section = inbound.caller if name == "caller" else getattr(inbound, name)
        _render_section(logger, section, name,
                        is_last_section=(i == len(inbound_sections) - 1),
                        indent="  │  ")

    _L(logger)
    _L(logger,  "  └─────────────────────────────────────────────────────────────")
    _L(logger)


def render_telemetry_summary(logger: logging.Logger, tel: PipelineTelemetry) -> None:
    """
    Clean numbered sequential telemetry report.

    Sections:
      1.  Total wall time
      2.  Phase durations
      3.  Feature amendment log  (chronological)
    """
    total_ns = time.perf_counter_ns() - (tel.pipeline_start_ns or 0)

    _L(logger)
    _L(logger, "  ┌─────────────────────────────────────────────────────────────")
    _L(logger, "  │  TELEMETRY SUMMARY")
    _L(logger)
    _L(logger, f"  │  1.  Total wall time")
    _L(logger, f"  │        {total_ns / 1_000_000:.3f} ms  ({total_ns:,} ns)")
    _L(logger)
    _L(logger,  "  │  2.  Phase durations")
    for name, ns in tel.phase_duration_ns.items():
        pct = (ns / total_ns * 100) if total_ns > 0 else 0
        _L(logger, f"  │        {name}")
        _L(logger, f"  │          {ns / 1_000_000:.3f} ms   {pct:.1f}% of total")
    _L(logger)
    _L(logger,  "  │  3.  Feature amendment log  (chronological)")
    for ms_t, step, feat, target, val in tel.feature_log:
        _L(logger, f"  │        [ {ms_t} ]  step {step}  ·  {feat}  →  {target}")
        _L(logger, f"  │          value  :  {val}")
    _L(logger)
    _L(logger, "  └─────────────────────────────────────────────────────────────")
    _L(logger)


# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
#  CONSOLE HELPERS
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

def _divider(label: str = "") -> None:
    if label:
        print(f"\n  ·  ·  ·  {label}  ·  ·  ·\n")
    else:
        print()

def _phase_header(number: int, title: str) -> None:
    print(f"\n┌────────────────────────────────────────────────────────────────")
    print(f"│  PHASE {number}  ·  {title}")
    print(f"│  {ms_now()}")
    print(f"│")

def _phase_footer(number: int, tel: PipelineTelemetry, phase_key: str) -> None:
    print(f"│  Duration  {tel.ms_duration(phase_key)}")
    print(f"└────────────────────────────────────────────────────────────────\n")

def _step(number: str, label: str) -> None:
    print(f"│  Step {number}  ·  {label}")

def _field(name: str, value: Any, indent: int = 2) -> None:
    pad = "│" + "    " * indent
    print(f"{pad}{name:<30}  {value}")

def _tandem_probe(jit: JITWorkObject, inbound: InboundRequestObject, label: str) -> None:
    print(f"│")
    print(f"│  [ PROBE ]  {label}")
    print(f"│  [ {ms_now()} ]")
    print(f"│    JIT     {obj_uid(jit)}")

    jit_sections  = [n for n in ("phase1","phase2","phase3","phase4") if getattr(jit, n) is not None]
    inb_sections  = [n for n in ("caller","phase1","phase2","phase3","phase4")
                     if getattr(inbound, n) is not None]
    print(f"│      sections : {jit_sections}")
    print(f"│    INBOUND {obj_uid(inbound)}")
    print(f"│      sections : {inb_sections}")
    print(f"│")

def _inter_phase(jit: JITWorkObject, inbound: InboundRequestObject,
                 leaving: str, entering: str) -> None:
    print(f"\n  ╌  ╌  ╌  ╌  ╌  exiting {leaving}  →  entering {entering}  ╌  ╌  ╌  ╌  ╌")
    print(f"  [ {ms_now()} ]")
    print(f"  JIT     {obj_uid(jit)}")
    print(f"  INBOUND {obj_uid(inbound)}\n")


# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
#
#  PHASE 1  ·  ADD ACQUISITION FEATURES
#  ─────────────────────────────────────
#  Appends Phase1Section onto both objects.
#
#  Steps:
#    1.1  Entry markers stamped on both
#    1.2  Inbound: pipeline_received  |  JIT: timezone_object
#    1.3  JIT acquires live UTC datetime from runtime clock
#    1.4  JIT converts UTC to local (Denver) datetime
#    1.5  JIT decomposes local datetime into discrete calendar fields
#    1.6  Exit markers stamped on both
#
#  ❶  PYTHON JIT NOTE — datetime.now() CALL specialization  (Python 3.11+)
#     Python 3.11 caches the C-level function pointer for datetime.now so
#     repeated calls skip Python attribute lookup overhead.
#     Enabled by: Python 3.11  (PEP 659 — Specializing Adaptive Interpreter)
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

def phase_1_add_acquisition_features(
    jit     : JITWorkObject,
    inbound : InboundRequestObject,
    tel     : PipelineTelemetry
) -> tuple[JITWorkObject, InboundRequestObject]:

    tel.phase_enter("Phase1_Acquisition")
    _phase_header(1, "Add Acquisition Features")

    # Both objects get a fresh Phase1Section appended
    jit.phase1     = Phase1Section()
    inbound.phase1 = Phase1Section()

    # ── 1.1  Entry markers ────────────────────────────────────────────────────
    _step("1.1", "Entry markers")
    entry_ms = ms_now()
    jit.phase1.entry_marker     = f"[ {entry_ms} ]  {obj_uid(jit)}"
    inbound.phase1.entry_marker = f"[ {entry_ms} ]  {obj_uid(inbound)}"
    tel.record("1.1", "entry_marker", "JIT",     jit.phase1.entry_marker)
    tel.record("1.1", "entry_marker", "Inbound", inbound.phase1.entry_marker)
    _field("JIT.phase1.entry_marker",     jit.phase1.entry_marker)
    _field("Inbound.phase1.entry_marker", inbound.phase1.entry_marker)

    _tandem_probe(jit, inbound, "Phase 1 ENTRY")

    # ── 1.2  Receipt confirmation + timezone ──────────────────────────────────
    _step("1.2", "pipeline_received → Inbound  |  timezone_object → JIT")
    inbound.phase1.pipeline_received = True
    jit.phase1.timezone_object       = pytz.timezone("America/Denver")
    tel.record("1.2", "pipeline_received", "Inbound", inbound.phase1.pipeline_received)
    tel.record("1.2", "timezone_object",   "JIT",     jit.phase1.timezone_object)
    _field("Inbound.phase1.pipeline_received", inbound.phase1.pipeline_received)
    _field("JIT.phase1.timezone_object",       jit.phase1.timezone_object)

    # ── 1.3  Live UTC datetime acquired by JIT ────────────────────────────────
    _step("1.3", "JIT acquires live UTC datetime from runtime clock")
    jit.phase1.raw_utc_datetime = datetime.datetime.now(pytz.utc)
    jit.phase1.utc_tzinfo       = str(jit.phase1.raw_utc_datetime.tzinfo)
    tel.record("1.3", "raw_utc_datetime", "JIT", jit.phase1.raw_utc_datetime)
    tel.record("1.3", "utc_tzinfo",       "JIT", jit.phase1.utc_tzinfo)
    _field("JIT.phase1.raw_utc_datetime", jit.phase1.raw_utc_datetime)
    _field("JIT.phase1.utc_tzinfo",       jit.phase1.utc_tzinfo)

    # ── 1.4  UTC converted to local (Denver) datetime ─────────────────────────
    _step("1.4", "JIT converts UTC → local (Denver) datetime")
    jit.phase1.raw_local_datetime = jit.phase1.raw_utc_datetime.astimezone(jit.phase1.timezone_object)
    jit.phase1.local_tzinfo       = str(jit.phase1.raw_local_datetime.tzinfo)
    jit.phase1.utc_offset         = jit.phase1.raw_local_datetime.utcoffset()
    tel.record("1.4", "raw_local_datetime", "JIT", jit.phase1.raw_local_datetime)
    tel.record("1.4", "utc_offset",         "JIT", jit.phase1.utc_offset)
    _field("JIT.phase1.raw_local_datetime", jit.phase1.raw_local_datetime)
    _field("JIT.phase1.local_tzinfo",       jit.phase1.local_tzinfo)
    _field("JIT.phase1.utc_offset",         jit.phase1.utc_offset)

    # ── 1.5  Decompose local datetime into discrete calendar fields ────────────
    _step("1.5", "JIT decomposes local datetime into calendar fields")
    ld = jit.phase1.raw_local_datetime
    jit.phase1.calendar_year     = ld.year
    jit.phase1.calendar_month    = ld.month
    jit.phase1.calendar_day      = ld.day
    jit.phase1.clock_hour        = ld.hour
    jit.phase1.clock_minute      = ld.minute
    jit.phase1.clock_second      = ld.second
    jit.phase1.clock_microsecond = ld.microsecond
    for f in ("calendar_year","calendar_month","calendar_day","clock_hour","clock_minute","clock_second","clock_microsecond"):
        tel.record("1.5", f, "JIT", getattr(jit.phase1, f))
    _field("date", f"{jit.phase1.calendar_year}-{jit.phase1.calendar_month:02d}-{jit.phase1.calendar_day:02d}")
    _field("time", f"{jit.phase1.clock_hour:02d}:{jit.phase1.clock_minute:02d}:{jit.phase1.clock_second:02d}.{jit.phase1.clock_microsecond}")

    # ── 1.6  Exit markers ─────────────────────────────────────────────────────
    _step("1.6", "Exit markers")
    exit_ms = ms_now()
    jit.phase1.exit_marker     = f"[ {exit_ms} ]  {obj_uid(jit)}"
    inbound.phase1.exit_marker = f"[ {exit_ms} ]  {obj_uid(inbound)}"
    tel.record("1.6", "exit_marker", "JIT",     jit.phase1.exit_marker)
    tel.record("1.6", "exit_marker", "Inbound", inbound.phase1.exit_marker)
    _field("JIT.phase1.exit_marker",     jit.phase1.exit_marker)
    _field("Inbound.phase1.exit_marker", inbound.phase1.exit_marker)

    _tandem_probe(jit, inbound, "Phase 1 EXIT")
    tel.phase_exit("Phase1_Acquisition")
    _phase_footer(1, tel, "Phase1_Acquisition")
    return jit, inbound


# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
#
#  PHASE 2  ·  ADD TRANSFORMATION FEATURES + TRANSFER
#  ───────────────────────────────────────────────────
#  Appends Phase2Section onto both objects.
#
#  Steps:
#    2.1  Entry markers
#    2.2  JIT formats GMT stamp from raw_utc_datetime
#    2.3  JIT formats local stamp from raw_local_datetime
#    2.4  Both stamps + human-readable datetimes transferred JIT → Inbound
#         transfer_marker stamped on both at exact ms of transfer
#    2.5  Exit markers
#
#  ❷  PYTHON JIT NOTE — f-string compilation  (Python 3.6+, 2016-12-23)
#     F-strings are compiled to a sequence of FORMAT_VALUE + BUILD_STRING
#     bytecodes.  Python 3.12+ inlines small string concatenations in the
#     specializing interpreter, avoiding intermediate list allocation.
#     Enabled by: Python 3.6 (PEP 498); inline optimization: Python 3.12
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

def phase_2_add_transformation_features_and_transfer(
    jit     : JITWorkObject,
    inbound : InboundRequestObject,
    tel     : PipelineTelemetry
) -> tuple[JITWorkObject, InboundRequestObject]:

    tel.phase_enter("Phase2_TransformAndTransfer")
    _phase_header(2, "Add Transformation Features + Transfer")

    jit.phase2     = Phase2Section()
    inbound.phase2 = Phase2Section()

    # ── 2.1  Entry markers ────────────────────────────────────────────────────
    _step("2.1", "Entry markers")
    entry_ms = ms_now()
    jit.phase2.entry_marker     = f"[ {entry_ms} ]  {obj_uid(jit)}"
    inbound.phase2.entry_marker = f"[ {entry_ms} ]  {obj_uid(inbound)}"
    tel.record("2.1", "entry_marker", "JIT",     jit.phase2.entry_marker)
    tel.record("2.1", "entry_marker", "Inbound", inbound.phase2.entry_marker)
    _field("JIT.phase2.entry_marker",     jit.phase2.entry_marker)
    _field("Inbound.phase2.entry_marker", inbound.phase2.entry_marker)

    _tandem_probe(jit, inbound, "Phase 2 ENTRY")

    # ── 2.2  JIT formats GMT stamp ────────────────────────────────────────────
    _step("2.2", "JIT formats GMT stamp")
    jit.phase2.formatted_gmt_stamp = jit.phase1.raw_utc_datetime.strftime("%H:%M:%S GMT %B %d, %Y")
    tel.record("2.2", "formatted_gmt_stamp", "JIT", jit.phase2.formatted_gmt_stamp)
    _field("JIT.phase2.formatted_gmt_stamp", jit.phase2.formatted_gmt_stamp)

    # ── 2.3  JIT formats local stamp ──────────────────────────────────────────
    _step("2.3", "JIT formats local stamp")
    ld = jit.phase1.raw_local_datetime
    jit.phase2.formatted_local_stamp = (
        f"{ld.hour % 12 or 12}:{ld.strftime('%M.%S.')}"
        f"{ld.strftime('%f')[:2]}.{ld.strftime('%f')[2:4]} "
        f"{ld.strftime('%p %B %d, %Y')}"
    )
    tel.record("2.3", "formatted_local_stamp", "JIT", jit.phase2.formatted_local_stamp)
    _field("JIT.phase2.formatted_local_stamp", jit.phase2.formatted_local_stamp)

    # ── 2.4  Transfer JIT → Inbound ───────────────────────────────────────────
    _step("2.4", "TRANSFER  JIT → Inbound  (formatted stamps + datetime strings)")
    transfer_ms = ms_now()
    jit.phase2.transfer_marker       = f"[ {transfer_ms} ]  {obj_uid(jit)}"
    inbound.phase2.transfer_marker   = f"[ {transfer_ms} ]  {obj_uid(inbound)}"
    inbound.phase2.formatted_gmt_stamp   = jit.phase2.formatted_gmt_stamp
    inbound.phase2.formatted_local_stamp = jit.phase2.formatted_local_stamp
    inbound.phase2.acquired_utc_str      = str(jit.phase1.raw_utc_datetime)
    inbound.phase2.acquired_local_str    = str(jit.phase1.raw_local_datetime)
    for feat, val in (
        ("transfer_marker",       inbound.phase2.transfer_marker),
        ("formatted_gmt_stamp",   inbound.phase2.formatted_gmt_stamp),
        ("formatted_local_stamp", inbound.phase2.formatted_local_stamp),
        ("acquired_utc_str",      inbound.phase2.acquired_utc_str),
        ("acquired_local_str",    inbound.phase2.acquired_local_str),
    ):
        tel.record("2.4", feat, "Inbound (from JIT)", val)
    _field("transfer_marker JIT",                 jit.phase2.transfer_marker)
    _field("transfer_marker Inbound",             inbound.phase2.transfer_marker)
    _field("Inbound.phase2.formatted_gmt_stamp",  inbound.phase2.formatted_gmt_stamp)
    _field("Inbound.phase2.formatted_local_stamp",inbound.phase2.formatted_local_stamp)
    _field("Inbound.phase2.acquired_utc_str",     inbound.phase2.acquired_utc_str)
    _field("Inbound.phase2.acquired_local_str",   inbound.phase2.acquired_local_str)

    # ── 2.5  Exit markers ─────────────────────────────────────────────────────
    _step("2.5", "Exit markers")
    exit_ms = ms_now()
    jit.phase2.exit_marker     = f"[ {exit_ms} ]  {obj_uid(jit)}"
    inbound.phase2.exit_marker = f"[ {exit_ms} ]  {obj_uid(inbound)}"
    tel.record("2.5", "exit_marker", "JIT",     jit.phase2.exit_marker)
    tel.record("2.5", "exit_marker", "Inbound", inbound.phase2.exit_marker)
    _field("JIT.phase2.exit_marker",     jit.phase2.exit_marker)
    _field("Inbound.phase2.exit_marker", inbound.phase2.exit_marker)

    _tandem_probe(jit, inbound, "Phase 2 EXIT")
    tel.phase_exit("Phase2_TransformAndTransfer")
    _phase_footer(2, tel, "Phase2_TransformAndTransfer")
    return jit, inbound


# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
#
#  PHASE 3  ·  ADD OUTPUT CONFIRMATION FEATURES
#  ─────────────────────────────────────────────
#  Appends Phase3Section onto both objects.
#
#  Steps:
#    3.1  Entry markers
#    3.2  Output from JIT displayed to console
#    3.3  Output from Inbound displayed to console (confirms transfer)
#    3.4  display_written / display_confirmed + shared display_marker
#    3.5  Exit markers
#
#  ❸  PYTHON JIT NOTE — bool attribute stores  (Python 3.12+)
#     Storing True on an object field.  Python 3.12 introduced immortal
#     True/False objects (PEP 683) meaning the STORE_ATTR for True skips
#     the reference-count increment/decrement entirely.
#     Enabled by: Python 3.12  (PEP 683 — immortal objects)
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

def phase_3_add_output_confirmation_features(
    jit     : JITWorkObject,
    inbound : InboundRequestObject,
    tel     : PipelineTelemetry
) -> tuple[JITWorkObject, InboundRequestObject]:

    tel.phase_enter("Phase3_OutputConfirmation")
    _phase_header(3, "Add Output Confirmation Features")

    jit.phase3     = Phase3Section()
    inbound.phase3 = Phase3Section()

    # ── 3.1  Entry markers ────────────────────────────────────────────────────
    _step("3.1", "Entry markers")
    entry_ms = ms_now()
    jit.phase3.entry_marker     = f"[ {entry_ms} ]  {obj_uid(jit)}"
    inbound.phase3.entry_marker = f"[ {entry_ms} ]  {obj_uid(inbound)}"
    tel.record("3.1", "entry_marker", "JIT",     jit.phase3.entry_marker)
    tel.record("3.1", "entry_marker", "Inbound", inbound.phase3.entry_marker)
    _field("JIT.phase3.entry_marker",     jit.phase3.entry_marker)
    _field("Inbound.phase3.entry_marker", inbound.phase3.entry_marker)

    _tandem_probe(jit, inbound, "Phase 3 ENTRY")

    # ── 3.2  Output from JIT ──────────────────────────────────────────────────
    _step("3.2", "Output from JITWorkObject")
    _field("[JIT]  GMT   ", jit.phase2.formatted_gmt_stamp)
    _field("[JIT]  Local ", jit.phase2.formatted_local_stamp)
    _field("[JIT]  date  ", f"{jit.phase1.calendar_year}-{jit.phase1.calendar_month:02d}-{jit.phase1.calendar_day:02d}")
    _field("[JIT]  time  ", f"{jit.phase1.clock_hour:02d}:{jit.phase1.clock_minute:02d}:{jit.phase1.clock_second:02d}.{jit.phase1.clock_microsecond}")

    # ── 3.3  Output from Inbound ──────────────────────────────────────────────
    _step("3.3", "Output from InboundRequestObject  (confirms transferred values)")
    _field(f"[{inbound.caller.request_label}]  GMT   ", inbound.phase2.formatted_gmt_stamp)
    _field(f"[{inbound.caller.request_label}]  Local ", inbound.phase2.formatted_local_stamp)
    _field(f"[{inbound.caller.request_label}]  UTC   ", inbound.phase2.acquired_utc_str)
    _field(f"[{inbound.caller.request_label}]  Local ", inbound.phase2.acquired_local_str)

    # ── 3.4  Display confirmation + shared display_marker ─────────────────────
    _step("3.4", "Display confirmation markers stamped on both")
    disp_ms = ms_now()
    jit.phase3.display_written    = True
    jit.phase3.display_marker     = f"[ {disp_ms} ]  {obj_uid(jit)}"
    inbound.phase3.display_confirmed = True
    inbound.phase3.display_marker    = f"[ {disp_ms} ]  {obj_uid(inbound)}"
    tel.record("3.4", "display_written",   "JIT",     jit.phase3.display_written)
    tel.record("3.4", "display_marker",    "JIT",     jit.phase3.display_marker)
    tel.record("3.4", "display_confirmed", "Inbound", inbound.phase3.display_confirmed)
    tel.record("3.4", "display_marker",    "Inbound", inbound.phase3.display_marker)
    _field("JIT.phase3.display_marker",     jit.phase3.display_marker)
    _field("Inbound.phase3.display_marker", inbound.phase3.display_marker)

    # ── 3.5  Exit markers ─────────────────────────────────────────────────────
    _step("3.5", "Exit markers")
    exit_ms = ms_now()
    jit.phase3.exit_marker     = f"[ {exit_ms} ]  {obj_uid(jit)}"
    inbound.phase3.exit_marker = f"[ {exit_ms} ]  {obj_uid(inbound)}"
    tel.record("3.5", "exit_marker", "JIT",     jit.phase3.exit_marker)
    tel.record("3.5", "exit_marker", "Inbound", inbound.phase3.exit_marker)
    _field("JIT.phase3.exit_marker",     jit.phase3.exit_marker)
    _field("Inbound.phase3.exit_marker", inbound.phase3.exit_marker)

    _tandem_probe(jit, inbound, "Phase 3 EXIT")
    tel.phase_exit("Phase3_OutputConfirmation")
    _phase_footer(3, tel, "Phase3_OutputConfirmation")
    return jit, inbound


# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
#
#  PHASE 4  ·  ADD LOG FEATURES + RELEASE JIT OBJECT
#  ──────────────────────────────────────────────────
#  Appends Phase4Section onto both objects.
#
#  Steps:
#    4.1  Entry markers
#    4.2  Logger configured; log_file_path added to both sections
#    4.3  Full hierarchical state of both objects written to log file
#         log_write_marker stamped on both at exact ms of write
#    4.4  log_confirmed added to Inbound
#    4.5  Full telemetry summary written to log file
#    4.6  release_marker — final feature JIT will ever carry
#    4.7  JITWorkObject manually destroyed via del + gc.collect()
#    4.8  exit_marker added to Inbound (JIT is gone)
#
#  ❹  PYTHON JIT NOTE — del + gc.collect()  (CPython all versions / gc: Python 2.0+)
#     del decrements ob_refcnt immediately.  If it reaches zero the object is
#     freed at that line.  gc.collect() sweeps reference cycles.
#     Python 3.12 immortal True/None skip refcount decrement (PEP 683).
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

def phase_4_add_log_features_and_release_jit_object(
    jit     : JITWorkObject,
    inbound : InboundRequestObject,
    tel     : PipelineTelemetry
) -> InboundRequestObject:

    tel.phase_enter("Phase4_LogAndRelease")
    _phase_header(4, "Add Log Features + Release JIT Object")

    jit.phase4     = Phase4Section()
    inbound.phase4 = Phase4Section()

    # ── 4.1  Entry markers ────────────────────────────────────────────────────
    _step("4.1", "Entry markers")
    entry_ms = ms_now()
    jit.phase4.entry_marker     = f"[ {entry_ms} ]  {obj_uid(jit)}"
    inbound.phase4.entry_marker = f"[ {entry_ms} ]  {obj_uid(inbound)}"
    tel.record("4.1", "entry_marker", "JIT",     jit.phase4.entry_marker)
    tel.record("4.1", "entry_marker", "Inbound", inbound.phase4.entry_marker)
    _field("JIT.phase4.entry_marker",     jit.phase4.entry_marker)
    _field("Inbound.phase4.entry_marker", inbound.phase4.entry_marker)

    _tandem_probe(jit, inbound, "Phase 4 ENTRY — all phases present on both objects")

    # ── 4.2  Logger configured; log paths added to both sections ──────────────
    _step("4.2", "Logger configured; log_file_path → both sections")
    log_file_path = os.path.join("/content", "time_factory_pipeline.log")
    logger = logging.getLogger("TimeFactoryPipeline")
    logger.setLevel(logging.DEBUG)
    if not logger.handlers:
        fmt = logging.Formatter("%(asctime)s  %(levelname)s  %(message)s")
        fh  = logging.FileHandler(log_file_path)
        fh.setLevel(logging.DEBUG)
        fh.setFormatter(fmt)
        ch  = logging.StreamHandler()
        ch.setLevel(logging.INFO)
        ch.setFormatter(fmt)
        logger.addHandler(fh)
        logger.addHandler(ch)
    jit.phase4.log_file_path     = log_file_path
    inbound.phase4.log_file_path = log_file_path
    tel.record("4.2", "log_file_path", "JIT",     jit.phase4.log_file_path)
    tel.record("4.2", "log_file_path", "Inbound", inbound.phase4.log_file_path)
    _field("JIT.phase4.log_file_path",     jit.phase4.log_file_path)
    _field("Inbound.phase4.log_file_path", inbound.phase4.log_file_path)

    # ── 4.3  Full hierarchical object state written to log file ───────────────
    _step("4.3", "Writing full hierarchical object state to log file")
    log_write_ms = ms_now()
    jit.phase4.log_write_marker     = f"[ {log_write_ms} ]  {obj_uid(jit)}"
    inbound.phase4.log_write_marker = f"[ {log_write_ms} ]  {obj_uid(inbound)}"
    tel.record("4.3", "log_write_marker", "JIT",     jit.phase4.log_write_marker)
    tel.record("4.3", "log_write_marker", "Inbound", inbound.phase4.log_write_marker)

    render_object_hierarchy(logger, jit, inbound, "Full Hierarchical Object State  ·  Phase 4")
    _field("JIT.phase4.log_write_marker",     jit.phase4.log_write_marker)
    _field("Inbound.phase4.log_write_marker", inbound.phase4.log_write_marker)

    # ── 4.4  log_confirmed added to Inbound ───────────────────────────────────
    _step("4.4", "log_confirmed → Inbound")
    inbound.phase4.log_confirmed = True
    tel.record("4.4", "log_confirmed", "Inbound", inbound.phase4.log_confirmed)
    _field("Inbound.phase4.log_confirmed", inbound.phase4.log_confirmed)

    # ── 4.5  Full telemetry summary written ───────────────────────────────────
    _step("4.5", "Emitting full telemetry summary")
    tel.phase_exit("Phase4_LogAndRelease")
    render_telemetry_summary(logger, tel)

    # ── 4.6  release_marker — final feature JIT will ever carry ───────────────
    _step("4.6", "release_marker — final feature added to JIT before deletion")
    release_ms = ms_now()
    jit.phase4.release_marker = f"[ {release_ms} ]  {obj_uid(jit)}"
    tel.record("4.6", "release_marker", "JIT", jit.phase4.release_marker)
    _field("JIT.phase4.release_marker", jit.phase4.release_marker)
    logger.info(f"  [ JIT ]  release_marker  :  {jit.phase4.release_marker}")
    logger.info(f"  [ JIT ]  ref_count before del  :  {ctypes.c_long.from_address(id(jit)).value}")

    _tandem_probe(jit, inbound, "PRE-RELEASE — final moment both objects exist together")

    # ── 4.7  JITWorkObject manually destroyed ─────────────────────────────────
    _step("4.7", "del jit + gc.collect()  —  JITWorkObject released from memory")
    _field("address before del",   hex(ctypes.addressof(ctypes.c_int.from_address(id(jit)))))
    _field("ref_count before del", ctypes.c_long.from_address(id(jit)).value)
    logger.info("  [ JIT ]  del jit_work_obj executing")
    del jit
    gc.collect()
    logger.info("  [ JIT ]  del + gc.collect() complete  —  JITWorkObject no longer in memory")
    _field("result", "JITWorkObject no longer in memory")

    # ── 4.8  Exit marker on Inbound (JIT is gone) ─────────────────────────────
    _step("4.8", "Exit marker on InboundRequestObject  (JIT destroyed, Inbound is sole survivor)")
    exit_ms = ms_now()
    inbound.phase4.exit_marker = f"[ {exit_ms} ]  {obj_uid(inbound)}"
    tel.record("4.8", "exit_marker", "Inbound", inbound.phase4.exit_marker)
    _field("Inbound.phase4.exit_marker", inbound.phase4.exit_marker)

    print(f"│")
    print(f"│  InboundRequestObject  {obj_uid(inbound)}")
    inb_sections = [n for n in ("caller","phase1","phase2","phase3","phase4") if getattr(inbound, n) is not None]
    print(f"│    sections : {inb_sections}")
    print(f"│  JITWorkObject  —  destroyed")
    print(f"│")

    _phase_footer(4, tel, "Phase4_LogAndRelease")
    return inbound


# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
#  FACTORY  ·  denver_time_factory
#
#  ❺  PYTHON JIT NOTE — type constructor specialization  (Python 3.10+)
#     JITWorkObject() created bare here and only here — JIT principle.
#     Python 3.10+ specializes the CALL to the type constructor, caching
#     tp_alloc + __init__ at C-level.
#     Enabled by: Python 3.10  (CALL specialization for type constructors)
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

def denver_time_factory(inbound_request: InboundRequestObject) -> InboundRequestObject:

    tel = PipelineTelemetry()
    tel.pipeline_start_ns = time.perf_counter_ns()

    print(f"\n┌────────────────────────────────────────────────────────────────")
    print(f"│  FACTORY  ·  denver_time_factory")
    print(f"│  [ {ms_now()} ]")
    print(f"│")
    print(f"│  InboundRequestObject received")
    print(f"│    {obj_uid(inbound_request)}")
    print(f"│    request_label  :  '{inbound_request.caller.request_label}'")
    inbound_request.telemetry = tel
    print(f"│  PipelineTelemetry attached")
    print(f"│")

    # JIT Work Object created bare at this exact moment
    jit_work_obj = JITWorkObject()
    print(f"│  JITWorkObject created bare")
    print(f"│    {obj_uid(jit_work_obj)}")
    print(f"│  Both objects entering tandem pipeline")
    print(f"│    JIT     {obj_uid(jit_work_obj)}")
    print(f"│    INBOUND {obj_uid(inbound_request)}")
    print(f"└────────────────────────────────────────────────────────────────\n")

    # Write the clean sequential header as the very first thing in the log file
    # (logger is not available yet — Phase 4 creates it.  We use a temporary
    #  bootstrap logger that writes only to the file so the header lands first.)
    _bootstrap_log_path = os.path.join("/content", "time_factory_pipeline.log")
    _bootstrap_logger   = logging.getLogger("TimeFactoryPipeline")
    if not _bootstrap_logger.handlers:
        _bootstrap_logger.setLevel(logging.DEBUG)
        _bfh = logging.FileHandler(_bootstrap_log_path)
        _bfh.setLevel(logging.DEBUG)
        _bfh.setFormatter(logging.Formatter("%(message)s"))
        _bootstrap_logger.addHandler(_bfh)
    render_log_header(_bootstrap_logger, jit_work_obj, inbound_request)

    jit_work_obj, inbound_request = phase_1_add_acquisition_features(jit_work_obj, inbound_request, tel)
    _inter_phase(jit_work_obj, inbound_request, "Phase 1", "Phase 2")

    jit_work_obj, inbound_request = phase_2_add_transformation_features_and_transfer(jit_work_obj, inbound_request, tel)
    _inter_phase(jit_work_obj, inbound_request, "Phase 2", "Phase 3")

    jit_work_obj, inbound_request = phase_3_add_output_confirmation_features(jit_work_obj, inbound_request, tel)
    _inter_phase(jit_work_obj, inbound_request, "Phase 3", "Phase 4")

    inbound_request = phase_4_add_log_features_and_release_jit_object(jit_work_obj, inbound_request, tel)

    print(f"\n┌────────────────────────────────────────────────────────────────")
    print(f"│  FACTORY  ·  Pipeline complete")
    print(f"│  [ {ms_now()} ]")
    print(f"│  JITWorkObject destroyed   InboundRequestObject returned")
    print(f"│    INBOUND {obj_uid(inbound_request)}")
    print(f"└────────────────────────────────────────────────────────────────\n")
    return inbound_request


# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
#  INVOCATION
#  TEMPLATE : populate caller section with your domain fields
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

print(f"\n┌────────────────────────────────────────────────────────────────")
print(f"│  INVOKE  ·  Invocation Begin")
print(f"│  [ {ms_now()} ]")
print(f"│")

inbound_request = InboundRequestObject()
inbound_request.caller.request_label = "DenverTimestampCapture-001"
inbound_request.caller.entry_marker  = stamp(inbound_request, "caller constructed")
# ↑  TEMPLATE : add your own domain fields to inbound_request.caller here

print(f"│  InboundRequestObject constructed")
print(f"│    {obj_uid(inbound_request)}")
print(f"│    caller.request_label  :  '{inbound_request.caller.request_label}'")
print(f"│    caller.entry_marker   :  {inbound_request.caller.entry_marker}")
print(f"└────────────────────────────────────────────────────────────────\n")

result = denver_time_factory(inbound_request)

print(f"\n┌────────────────────────────────────────────────────────────────")
print(f"│  INVOKE  ·  Result")
print(f"│  [ {ms_now()} ]")
print(f"│    {obj_uid(result)}")
print(f"│")
print(f"│  caller")
print(f"│    request_label          :  {result.caller.request_label}")
print(f"│    entry_marker           :  {result.caller.entry_marker}")
print(f"│")
print(f"│  phase1")
print(f"│    entry_marker           :  {result.phase1.entry_marker}")
print(f"│    pipeline_received      :  {result.phase1.pipeline_received}")
print(f"│    exit_marker            :  {result.phase1.exit_marker}")
print(f"│")
print(f"│  phase2")
print(f"│    entry_marker           :  {result.phase2.entry_marker}")
print(f"│    formatted_gmt_stamp    :  {result.phase2.formatted_gmt_stamp}")
print(f"│    formatted_local_stamp  :  {result.phase2.formatted_local_stamp}")
print(f"│    transfer_marker        :  {result.phase2.transfer_marker}")
print(f"│    acquired_utc_str       :  {result.phase2.acquired_utc_str}")
print(f"│    acquired_local_str     :  {result.phase2.acquired_local_str}")
print(f"│    exit_marker            :  {result.phase2.exit_marker}")
print(f"│")
print(f"│  phase3")
print(f"│    entry_marker           :  {result.phase3.entry_marker}")
print(f"│    display_confirmed      :  {result.phase3.display_confirmed}")
print(f"│    display_marker         :  {result.phase3.display_marker}")
print(f"│    exit_marker            :  {result.phase3.exit_marker}")
print(f"│")
print(f"│  phase4")
print(f"│    entry_marker           :  {result.phase4.entry_marker}")
print(f"│    log_file_path          :  {result.phase4.log_file_path}")
print(f"│    log_write_marker       :  {result.phase4.log_write_marker}")
print(f"│    log_confirmed          :  {result.phase4.log_confirmed}")
print(f"│    exit_marker            :  {result.phase4.exit_marker}")
print(f"└────────────────────────────────────────────────────────────────")
print(f"  INVOKE  ·  Complete  [ {ms_now()} ]")
print()

2026-03-03 20:26:01,707  TimeFactoryPipeline  INFO  
INFO:TimeFactoryPipeline:
2026-03-03 20:26:01,709  TimeFactoryPipeline  INFO    ╔══════════════════════════════════════════════════════════════
INFO:TimeFactoryPipeline:  ╔══════════════════════════════════════════════════════════════
2026-03-03 20:26:01,712  TimeFactoryPipeline  INFO    ║  PIPELINE EXECUTION LOG
INFO:TimeFactoryPipeline:  ║  PIPELINE EXECUTION LOG
2026-03-03 20:26:01,713  TimeFactoryPipeline  INFO    ║  2026-03-03  20:26:01.713
INFO:TimeFactoryPipeline:  ║  2026-03-03  20:26:01.713
2026-03-03 20:26:01,715  TimeFactoryPipeline  INFO    ╠══════════════════════════════════════════════════════════════
INFO:TimeFactoryPipeline:  ╠══════════════════════════════════════════════════════════════
2026-03-03 20:26:01,716  TimeFactoryPipeline  INFO  
INFO:TimeFactoryPipeline:
2026-03-03 20:26:01,717  TimeFactoryPipeline  INFO    ║  1.  What was passed in
INFO:TimeFactoryPipeline:  ║  1.  What was passed in
2026-03-03 20:26:01,7


┌────────────────────────────────────────────────────────────────
│  INVOKE  ·  Invocation Begin
│  [ 2026-03-03  20:26:01.707 ]
│
│  InboundRequestObject constructed
│    OBJ#0x7f16d4894e00@2
│    caller.request_label  :  'DenverTimestampCapture-001'
│    caller.entry_marker   :  [ 2026-03-03  20:26:01.707 ]  OBJ#0x7f16d4894e00@3  caller constructed
└────────────────────────────────────────────────────────────────


┌────────────────────────────────────────────────────────────────
│  FACTORY  ·  denver_time_factory
│  [ 2026-03-03  20:26:01.707 ]
│
│  InboundRequestObject received
│    OBJ#0x7f16d4894e00@3
│    request_label  :  'DenverTimestampCapture-001'
│  PipelineTelemetry attached
│
│  JITWorkObject created bare
│    OBJ#0x7f16bfd2c800@2
│  Both objects entering tandem pipeline
│    JIT     OBJ#0x7f16bfd2c800@2
│    INBOUND OBJ#0x7f16d4894e00@3
└────────────────────────────────────────────────────────────────


  ╔══════════════════════════════════════════════════════════════
 

2026-03-03 20:26:01,909  TimeFactoryPipeline  INFO    │          value  :  20:26:01 GMT March 03, 2026
INFO:TimeFactoryPipeline:  │          value  :  20:26:01 GMT March 03, 2026
2026-03-03 20:26:01,910  TimeFactoryPipeline  INFO    │        [ 2026-03-03  20:26:01.742 ]  step 2.3  ·  formatted_local_stamp  →  JIT
INFO:TimeFactoryPipeline:  │        [ 2026-03-03  20:26:01.742 ]  step 2.3  ·  formatted_local_stamp  →  JIT
2026-03-03 20:26:01,911  TimeFactoryPipeline  INFO    │          value  :  1:26.01.74.16 PM March 03, 2026
INFO:TimeFactoryPipeline:  │          value  :  1:26.01.74.16 PM March 03, 2026
2026-03-03 20:26:01,912  TimeFactoryPipeline  INFO    │        [ 2026-03-03  20:26:01.742 ]  step 2.4  ·  transfer_marker  →  Inbound (from JIT)
INFO:TimeFactoryPipeline:  │        [ 2026-03-03  20:26:01.742 ]  step 2.4  ·  transfer_marker  →  Inbound (from JIT)
2026-03-03 20:26:01,913  TimeFactoryPipeline  INFO    │          value  :  [ 2026-03-03  20:26:01.742 ]  OBJ#0x7f16d4894e00@4


  │          value  :  20:26:01 GMT March 03, 2026
  │        [ 2026-03-03  20:26:01.742 ]  step 2.3  ·  formatted_local_stamp  →  JIT
  │          value  :  1:26.01.74.16 PM March 03, 2026
  │        [ 2026-03-03  20:26:01.742 ]  step 2.4  ·  transfer_marker  →  Inbound (from JIT)
  │          value  :  [ 2026-03-03  20:26:01.742 ]  OBJ#0x7f16d4894e00@4
  │        [ 2026-03-03  20:26:01.742 ]  step 2.4  ·  formatted_gmt_stamp  →  Inbound (from JIT)
  │          value  :  20:26:01 GMT March 03, 2026
  │        [ 2026-03-03  20:26:01.742 ]  step 2.4  ·  formatted_local_stamp  →  Inbound (from JIT)
  │          value  :  1:26.01.74.16 PM March 03, 2026
  │        [ 2026-03-03  20:26:01.742 ]  step 2.4  ·  acquired_utc_str  →  Inbound (from JIT)
  │          value  :  2026-03-03 20:26:01.741615+00:00
  │        [ 2026-03-03  20:26:01.742 ]  step 2.4  ·  acquired_local_str  →  Inbound (from JIT)
  │          value  :  2026-03-03 13:26:01.741615-07:00
  │        [ 2026-03-03  20:26:01.742 ]